# OOP Part 2 — Encapsulation & Static

**Topic:** 04.02 · **Level:** Intermediate · **Type:** THEORY + CODE

## Welcome back to OOP

In Session 7 you learned to define classes and create objects. Now you will make objects *interact* and *protect themselves*. A real program is a community of objects: a `Point` talks to another `Point`, a `Person` greets you, an `Atm` guards its balance, and a bowl of scoops tracks a total price. This session gives you the tools those communities rely on.

## What you will learn

1. Accessing attributes and methods of objects
2. Reference variables & pass-by-reference
3. Encapsulation — private attributes
4. Collection of objects
5. Static variables & static methods

## The scenario

By the end you will be able to write an OOP solution to this concrete problem set:

- A user can create and view 2D coordinates.
- A user can find the distance between two coordinates.
- A user can find the distance of a coordinate from the origin.
- A user can check whether a point lies on a given line.
- A user can find the distance between a 2D point and a given line.

These map beautifully onto two classes: `Point` (its own coordinates plus distance logic) and `Line` (a straight line `Ax + By + C = 0` plus point-line checks). That design keeps the math inside the object that owns the data.

## Key concepts preview

- **Reference variables**: an object lives in memory; a variable only *points* at it, and many variables can point at the same object.
- **Pass by reference**: when you hand an object to a function, the function mutates the very same object — changes are visible outside.
- **Encapsulation**: hiding an object's data behind methods with the `__` prefix so outside code cannot touch it directly.
- **Static members**: class-level variables and `@staticmethod` methods shared by all instances, usable even without an object.

Objects then become not just data holders but *responsible citizens* that expose a clean public interface and keep their own private state consistent. Let's begin with the two geometry classes.

# 1. Modeling a Problem with Classes

## 1.1 Point and Line

Object-Oriented Programming shines when you name the real things in your problem and give each one its data plus its behavior. For the geometry scenario, the nouns are **Point** and **Line**.

```python
class Point:
    def __init__(self, x, y):
        self.x_cod = x
        self.y_cod = y
```

- `Point` stores its coordinates (`x_cod`, `y_cod`) as instance attributes.
- `__str__` makes `print(p1)` show `<1,10>` instead of a memory address.
- `euclidean_distance(other)` applies the distance formula `sqrt((x1-x2)^2 + (y1-y2)^2)` between two points.
- `distance_from_origin()` computes `sqrt(x^2 + y^2)` — the commented alternative `return self.euclidean_distance(Point(0,0))` reuses one method inside another.

`Line` stores the coefficients `A`, `B`, `C` of the line `Ax + By + C = 0`. `point_on_line(line, point)` plugs the point's coordinates into the line equation and checks whether the result is zero. `shortest_distance(line, point)` applies the point-to-line formula `|A*x + B*y + C| / sqrt(A^2 + B^2)`.

Notice that these two methods name their parameters `line` and `point` instead of the conventional `self` — it still works because the instance is simply passed as the first argument, but later sections show the idiomatic `self` style. The payoff of this design: the main code collapses to `l1 = Line(1,1,-2)`, `p1 = Point(1,10)`, and a call such as `l1.shortest_distance(p1)` → `6.36`.

In [1]:
class Point:

  def __init__(self,x,y):
    self.x_cod = x
    self.y_cod = y

  def __str__(self):
    return '<{},{}>'.format(self.x_cod,self.y_cod)

  def euclidean_distance(self,other):
    return ((self.x_cod - other.x_cod)**2 + (self.y_cod - other.y_cod)**2)**0.5

  def distance_from_origin(self):
    return (self.x_cod**2 + self.y_cod**2)**0.5
    # return self.euclidean_distance(Point(0,0))


class Line:

  def __init__(self,A,B,C):
    self.A = A
    self.B = B
    self.C = C

  def __str__(self):
    return '{}x + {}y + {} = 0'.format(self.A,self.B,self.C)

  def point_on_line(line,point):
    if line.A*point.x_cod + line.B*point.y_cod + line.C == 0:
      return "lies on the line"
    else:
      return "does not lie on the line"

  def shortest_distance(line,point):
    return abs(line.A*point.x_cod + line.B*point.y_cod + line.C)/(line.A**2 + line.B**2)**0.5


In [2]:
l1 = Line(1,1,-2)
p1 = Point(1,10)
print(l1)
print(p1)

l1.shortest_distance(p1)

1x + 1y + -2 = 0
<1,10>


6.363961030678928

# 2. Accessing Object Attributes & Methods

## 2.1 Attributes are data, methods are actions

Once an object exists, you read its pieces and call its powers with the **dot operator**:

```python
p = Person('nitish', 'india')
p.name       # ATTRIBUTE — data property, no brackets
p.greet()    # METHOD — callable action, brackets needed
```

- `obj.attribute` reads a piece of data — the dot reaches into the object's namespace.
- `obj.method()` runs a function bound to the object — the parentheses are required.
- The constructor stored the values into `self` inside `__init__`; the dot retrieves them.
- Different objects of the same class return different attribute values — state is per-instance.
- Attributes are public by default in Python.
- Forgetting the `()` on a method gives you the function object itself, not its result.

This is the everyday syntax of OOP — you will write `p.name` and `p.greet()` constantly from here on. Spot the `Person` class in the first cell, then try both accesses in the cells after it.

### How objects access attributes

### Key ideas
Classes/objects model state+behavior. Encapsulation hides internals; abstraction focuses on essentials. Inheritance/polymorphism extend reuse.

In [3]:
class Person:

  def __init__(self,name_input,country_input):
    self.name = name_input
    self.country = country_input

  def greet(self):
    if self.country == 'india':
      print('Namaste',self.name)
    else:
      print('Hello',self.name)


## 2.2 Deep dive — attribute access mechanics

```python
class Person:
    def __init__(self, name_input, country_input):
        self.name = name_input
        self.country = country_input

    def greet(self):
        if self.country == 'india':
            print('Namaste', self.name)
        else:
            print('Hello', self.name)

p = Person('nitish', 'india')
p.name        # 'nitish' — attribute
p.greet()     # Namaste nitish — method
```

- Attribute: `obj.attribute` — a data property.
- Method: `obj.method()` — callable, needs brackets.
- The constructor stores the incoming values into `self`, the object's own attribute dictionary.
- The dot syntax accesses that namespace.
- A method without `()` returns a bound function reference (its identity), not the result.
- Outside access to attributes is public by default.

The `Person` class below greets with `Namaste` when the country is `india`, and `Hello` otherwise — a tidy example of behavior decided by the object's own state. Notice the constructor parameter names `name_input` and `country_input`: they avoid shadowing `self.name` and make the storage step self-documenting. Get comfortable associating each `obj.attribute` access with the line in `__init__` that created it.

In [4]:
# how to access attributes
p = Person('nitish','india')

In [5]:
p.name

'nitish'

In [6]:
# how to access methods
p.greet()

Namaste nitish


# 3. Non-existent Attributes

## 3.1 You cannot read what the object never got

An object only carries the attributes you actually created for it — usually inside `__init__`:

```python
p.gender   # AttributeError: 'Person' object has no attribute 'gender'
```

If a class never set `self.gender`, then `p.gender` fails with an `AttributeError`, and the message tells you exactly which name is missing and on which class.

| Guard | What it does |
|---|---|
| `hasattr(p, 'gender')` | checks before accessing |
| `getattr(p, 'gender', None)` | safe read with a default |

The fix is almost always: define that attribute in `__init__` first. This error is one of the most common beginner mistakes — whenever it appears, ask yourself "did I set this on `self`?". In the code cell right below, `p.gender` raises the error on purpose; notice how Python names both the class and the missing attribute in its message, so debugging is mostly a matter of reading the traceback.

## Non-existent attribute

```python
p.gender    # AttributeError! class ne ye attribute define hi nahi kiya
```

- An attribute that was not created with `self.` in `__init__` cannot be accessed
- `AttributeError: 'Person' object has no attribute 'gender'`
- Fix: define that attribute in the class's `__init__` first


## 3.2 Deep dive — AttributeError in practice

```python
p.gender  # AttributeError: 'Person' object has no attribute 'gender'
```

- Accessing an attribute that was never set on the object raises `AttributeError`.
- The message names the exact class and attribute: `'Person' object has no attribute 'gender'`.
- The object only holds what `__init__` (or a later assignment) created.
- Defensive reading: `hasattr(p, 'gender')` first, or `getattr(p, 'gender', None)`.
- Attributes are created at runtime — Python objects are dynamic.
- "Set-before-access" is the typical bug; read the class and the assignment line when debugging.

Since `gender` was never stored on `self`, the cell below raises the error on purpose — a small, instructive crash that makes the rule unforgettable. The error also teaches the language of tracebacks: read the bottom line first — `AttributeError`, then the object type, then the exact missing name. Nine times out of ten the fix is one line back in `__init__`, where the attribute should have been created.

In [7]:
# what if i try to access non-existent attributes
p.gender

AttributeError: 'Person' object has no attribute 'gender'

# 4. Creating Attributes from Outside

## 4.1 Python objects are flexible

Unlike strict languages, Python lets you bolt new attributes onto an object from outside its class:

```python
p.gender = 'male'     # created AFTER construction
print(p.gender)       # 'male'
```

- The assignment simply adds `gender` to *that particular object's* namespace.
- Only that object changes — other instances are untouched.
- Nothing stops you; there is no compile-time check.

| Approach | Effect |
|---|---|
| Inside `__init__` | consistent, guaranteed for every object |
| Outside (ad-hoc) | only for one object — handy for prototyping |

Best practice: define all normal attributes in `__init__` for consistency, and save outside assignment for quick experiments and optional flags. You will add `p.gender = 'male'` in the cells below and see it come back instantly — which raises a design question worth answering now: *should* a class's every field be declared up front? For consistency, yes; for prototyping, the dynamic way is often faster.

### Attribute creation from outside of the class

### Key ideas
Classes/objects model state+behavior. Encapsulation hides internals; abstraction focuses on essentials. Inheritance/polymorphism extend reuse.

## Creating an attribute from outside the class

```python
p.gender = 'male'   # class ke bahar se object pe attribute add
print(p.gender)     # 'male'
```

- The Python objects are flexible — attributes can be added to an object dynamically
- This only affects that particular object
- For consistency, define all attributes in `__init__`


## 4.2 Deep dive — the dynamic instance namespace

```python
p.gender = 'male'    # new attribute added without touching the class
p.gender             # 'male'
```

- Python objects are dynamic: `obj.new_attr = value` updates the instance namespace from anywhere.
- There is no real encapsulation by default — everything is public at runtime.
- The change affects ONLY that object, not other instances.
- Because it was never declared in `__init__`, it is an *optional* field.
- Use it for ad-hoc flags and quick prototyping.
- Risk: inconsistency — two objects of the same class can disagree on which attributes they have.
- Mental model: every instance has its own attribute dictionary you are free to edit.

Run the two cells below: `p.gender = 'male'` adds the attribute, and `p.gender` returns it. The class definition never mentioned `gender` — Python simply stored it on this one object. This is the flip side of Python's famous "everything is public": total freedom, and the risk that two instances drift apart. Decide deliberately — declared fields in `__init__` for the stable contract, dynamic attributes for quick experiments — and your code keeps both virtues.

In [8]:
p.gender = 'male'

In [9]:
p.gender

'male'

# 5. Reference Variables

## 5.1 The name is a pointer, not the object

When you create an object, Python sets aside memory for it; the variable is just a *label* attached to that memory:

```python
p = Person()   # object created, labeled `p`
q = p          # q points at the SAME object
print(id(p), id(q))   # identical addresses
```

- A reference variable does not contain the object — it holds its address.
- An object created without a reference (`Person()` with nothing on the left) becomes unreachable and is garbage-collected.
- One object can wear many labels: `q = p` creates NO new object, just a second name for the same one.
- Changing `q.name` changes what `p` sees, because both names reach the same memory.

This is why `id(p) == id(q)` is `True` after `q = p` — the label swap did not stamp out a copy.

### Reference Variables

- Reference variables hold the objects
- We can create objects without reference variable as well
- An object can have multiple reference variables
- Assigning a new reference variable to an existing object does not create a new object

## Reference variables

```python
class Person:
    def __init__(self):
        self.name = 'nitish'
        self.gender = 'male'

p = Person()    # object is referenced by label `p`
q = p           # q bhi same object ko point karega!
```

- An object created without a reference gets lost in memory
- An object can have multiple reference variables
- `q = p` does NOT create a new object — same object, 2 labels
- Multiple refs → `print(id(p))`, `print(id(q))` give the same address


## 5.2 Deep dive — object lifecycle & multiple references

```python
class Person:
    def __init__(self):
        print('Created')

Person()        # object created — but no reference stored!
```

- `Person()` without assignment → the object is born, but no label keeps it alive.
- Reference count drops to 0 → **garbage collected** immediately after `'Created'` prints.
- A reference variable is a tag pointing at the object.
- Multiple references: `q = p` — same object, two names; `id(p) == id(q)` confirms it.
- Changing an attribute through `q` is visible through `p` — same underlying object.
- Rebinding a label (`p = something_else`) redirects only that name; the old object lives while another label still points to it.
- Python manages memory for you; later you may meet `del`, `weakref`, and the `gc` module for edge cases.

The cells below create a `Person`, alias it as `q`, print matching ids, and then change `q.name` — demonstrating that `p` sees the change too to `ankit`.

In [10]:
# object without a reference
class Person:

  def __init__(self):
    self.name = 'nitish'
    self.gender = 'male'

p = Person()
q = p

In [11]:
# Multiple ref
print(id(p))
print(id(q))

2884150648448
2884150648448


In [12]:
# change attribute value with the help of 2nd object

## Deep Dive : Multiple references — same object

```python
p = Person('nitish')      # 1 object
q = p                     # q same object ko point
id(p) == id(q)            # True — SAME object
q.name = 'ankit'          # usi object ka name change
print(p.name)             # 'ankit' — p bhi dekh lega!
```

- `q = p` — assignment of a REFERENCE, not a copy
- Both names → the same memory
- Change via `q` → visible on `p` (same underlying object)
- id verification: `id(p) == id(q)` True
- This is the fundamental difference between primitive-immutable and object-mutable
- If you don't want sharing — use `copy()`, `deepcopy()`
- Aliasing concept — happens more with collections of objects
- Interview: "does p change when q changes?"


In [13]:
print(p.name)
print(q.name)
q.name = 'ankit'
print(q.name)
print(p.name)

nitish
nitish
ankit
ankit


# 6. Pass by Reference

## 6.1 What goes into the function, comes out changed

Functions receive objects **by reference** — the same memory, not a copy:

```python
def greet(person):
    person.name = 'ankit'   # mutates the caller's object
    print(person.name)

p = Person('nitish', 'male')
greet(p)                    # prints 'ankit'
print(p.name)               # also 'ankit' — the original changed!
```

- Object parameters are references; attribute changes inside the function leak outside.
- Immutables (int, str, tuple) get *rebound*, so reassigning a parameter inside the function never affects the caller.
- Mutables (list, dict, your objects) get *mutated*, so in-place changes are visible outside.
- If the caller must stay unchanged, pass or work on a copy.

The root cause is the aliasing you already saw with `q = p` — labels `p` and `person` point to one shared object, now crossing a function boundary. The cells below demonstrate both flavours: one function reads the object and builds a brand-new one to return, while another mutates the passed-in object, proving the distinction between "outside" data and "shared" data.

### Pass by reference

## Pass by reference — the change inside is visible outside

```python
def greet(person):         # person object ka reference hai
    person.name = 'ankit'  # original object change!
    print(person.name)

p = Person('nitish', 'male')
greet(p)
print(p.name)              # 'ankit' — same object andar gya
```

- In Python objects are passed **by reference**, not by copy
- Attribute change inside the function → visible on the caller side too
- Strings/immutables get copied, objects do not


## 6.2 Deep dive — mutation across the call boundary

```python
class Person:
    def __init__(self, name, gender):
        self.name = name

def greet(person):
    person.name = 'changed'   # mutates the shared object

p = Person('nitish', 'male')
greet(p)
print(p.name)      # 'changed' — the caller's object changed too!
```

- Objects are passed **by reference** into functions.
- Mutation inside → the caller's object is affected: same object, shared memory.
- Immutables (int/str): rebinding inside the function is invisible outside.
- Mutables (list/dict/object): in-place changes are visible outside.
- This causes classic bugs where a function silently rewrites caller data.
- Defensive approach: copy the object inside the function first.
- Mental model: variables are arrows, objects are boxes; function parameters copy the arrow, not the box.

The cells below show it live: `greet(p)` receives the same id as `p`, sets `person.name = 'ankit'`, and `print(p.name)` prints `ankit` — proof the change happened on the original object.

In [14]:
class Person:

  def __init__(self,name,gender):
    self.name = name
    self.gender = gender

# outside the class -> function
def greet(person):
  print('Hi my name is',person.name,'and I am a',person.gender)
  p1 = Person('ankit','male')
  return p1

p = Person('nitish','male')
x = greet(p)
print(x.name)
print(x.gender)

Hi my name is nitish and I am a male
ankit
male


In [15]:
class Person:

  def __init__(self,name,gender):
    self.name = name
    self.gender = gender

# outside the class -> function
def greet(person):
  print(id(person))
  person.name = 'ankit'
  print(person.name)

p = Person('nitish','male')
print(id(p))
greet(p)
print(p.name)

2884150649120
2884150649120
ankit
ankit


# 7. Object Mutability

## 7.1 Mutable objects change in place

Because objects are mutable, any method or function holding a reference can modify them:

```python
def greet(person):
    person.name = 'ankit'   # edits the shared object
    return person

p = Person('nitish', 'male')
q = greet(p)
print(p.name, q.name)       # both 'ankit' — same object
```

- `person.name = ...` modifies the object no matter which reference performed the write.
- The function's `return person` hands back the SAME object, so `q` and `p` point to one thing.
- This mirrors list behavior you already know: `append` inside a function changes the list outside.

The opposite is immutability: an integer like `5` can never change in place — `x = x + 1` *rebinds* `x` to a brand-new value, leaving the old one alone. Keep that contrast in your mental model: immutables are replaced, mutables are edited — and your objects belong firmly to the editable camp.

### Object mutability


### Key ideas
Classes/objects model state+behavior. Encapsulation hides internals; abstraction focuses on essentials. Inheritance/polymorphism extend reuse.

## 7.2 Deep dive — mutable state everywhere

```python
def greet(person):
    person.name = 'ankit'
    return person

p = Person('nitish', 'male')
p1 = greet(p)
print(id(p))     # some address
print(id(p1))    # SAME address — the function returned the same object
```

- Objects are **mutable** — methods and functions can modify them in place.
- `self.name = ...` works on the same object no matter which method or function runs.
- `return person` returns a reference, not a copy — `p` and `p1` share memory.
- The two ids print equal, confirming identity.
- This is the same feature you used when mutating lists.

Run the cell and watch `id(p) == id(p1)`. If you ever want independent copies, reach for `copy()` or `copy.deepcopy()` — topics for the collections chapters. Mutation and side effects are not bugs by themselves — they are how objects cooperate — but they demand awareness: if a function unexpectedly rewrites your data, ask who else holds a reference. Aliasing is the root cause of a whole family of "why did it change?" mysteries.

In [16]:
class Person:

  def __init__(self,name,gender):
    self.name = name
    self.gender = gender

# outside the class -> function
def greet(person):
  person.name = 'ankit'
  return person

p = Person('nitish','male')
print(id(p))
p1 = greet(p)
print(id(p1))

2884150642736
2884150642736


# 8. Encapsulation

## 8.1 Private data behind public gates

**Encapsulation** hides an object's internal state behind methods so outside code cannot pounce on it directly:

```python
class Atm:
    def __init__(self):
        self.pin = ''          # public attribute
        self.__balance = 0     # private — __ prefix

    def get_balance(self):
        return self.__balance   # gatekeeper (read)

    def set_balance(self, value):
        if type(value) == int:
            self.__balance = value   # gatekeeper (write, with validation)
```

- `__name` (double underscore) marks an attribute **private** via name mangling.
- Direct access from outside fails; code must use the methods.
- Getters read, setters write, and setters can **validate** — e.g. accept only integers.
- Benefits: change internal logic later without breaking callers, and keep state consistent (invariants).

Privacy is a strong convention, not a hard wall — the mangled name `_Atm__balance` still technically works, but touching it is discouraged. Enforce the contract with a public method and private storage; the reader of any other class then knows instantly which names are "for internal use" without reading documentation.

### Encapsulation

## 8.2 Deep dive — name mangling & private members

```python
class Atm:
    __counter = 1                 # static variable too

    def __init__(self):
        self.__pin = 1234          # NAME-MANGLED private attribute
    def get_pin(self):             # getter
        return self.__pin
    def set_pin(self, new_pin):    # setter
        if len(str(new_pin)) == 4:
            self.__pin = new_pin
```

- `__var` (double underscore) marks an attribute **private**; Python *name-mangles* it.
- What is really stored is `_Atm__pin` — technically reachable, but by convention private.
- Getter to read (`get_balance`), setter to write (`set_balance`).
- The setter can validate: here, a pin must be four digits.
- Encapsulation protects invariants — an account balance should never be changed at will.
- Public interface = methods; private = data.
- Not enforced by the language (unlike Java `private`) — it is a strong convention.

The `Atm` class in the cells below puts it into practice with a guarded balance and validate-before-write setters. Start with your own raw `__atm_pin` readable, too — then comment it out and watch the `AttributeError` appear: the mangling is doing its job the moment the name stops resolving.

In [17]:
# instance var -> python tutor
class Person:

  def __init__(self,name_input,country_input):
    self.name = name_input
    self.country = country_input

p1 = Person('nitish','india')
p2 = Person('steve','australia')

In [18]:
p2.name

'steve'

## 8.3 The guarded Atm at work

```python
class Atm:
    def __init__(self):
        print(id(self))
        self.pin = ''
        self.__balance = 0

    def get_balance(self):
        return self.__balance

    def set_balance(self, new_value):
        if type(new_value) == int:
            self.__balance = new_value
        else:
            print('beta bahot maarenge')
```

The full `Atm` in the upcoming cells wraps its balance in `__balance` and exposes `get_balance()` / `set_balance()`. `set_balance` refuses non-integer values — validation lives where the write happens. The menus, pin creation and withdrawal logic all work through the same private state, so no outside code can corrupt the balance.

The test cells that follow do the essentials: `obj = Atm()` prints the object id from inside the constructor; `obj.get_balance()` reads through the getter; `obj.set_balance(1000)` writes through the validating setter; and the interactive `obj.withdraw()` asks for the pin and amount, then deducts from the private `__balance` when the amount is affordable. What matters is the pattern — private state plus public, validated behavior.

In [19]:
class Atm:

  # constructor(special function)->superpower -> 
  def __init__(self):
    print(id(self))
    self.pin = ''
    self.__balance = 0
    #self.menu()

  def get_balance(self):
    return self.__balance

  def set_balance(self,new_value):
    if type(new_value) == int:
      self.__balance = new_value
    else:
      print('beta bahot maarenge')

  def __menu(self):
    user_input = input("""
    Hi how can I help you?
    1. Press 1 to create pin
    2. Press 2 to change pin
    3. Press 3 to check balance
    4. Press 4 to withdraw
    5. Anything else to exit
    """)

    if user_input == '1':
      self.create_pin()
    elif user_input == '2':
      self.change_pin()
    elif user_input == '3':
      self.check_balance()
    elif user_input == '4':
      self.withdraw()
    else:
      exit()

  def create_pin(self):
    user_pin = input('enter your pin')
    self.pin = user_pin

    user_balance = int(input('enter balance'))
    self.__balance = user_balance

    print('pin created successfully')

  def change_pin(self):
    old_pin = input('enter old pin')

    if old_pin == self.pin:
      # let him change the pin
      new_pin = input('enter new pin')
      self.pin = new_pin
      print('pin change successful')
    else:
      print('nai karne de sakta re baba')

  def check_balance(self):
    user_pin = input('enter your pin')
    if user_pin == self.pin:
      print('your balance is ',self.__balance)
    else:
      print('chal nikal yahan se')

  def withdraw(self):
    user_pin = input('enter the pin')
    if user_pin == self.pin:
      # allow to withdraw
      amount = int(input('enter the amount'))
      if amount <= self.__balance:
        self.__balance = self.__balance - amount
        print('withdrawl successful.balance is',self.__balance)
      else:
        print('abe garib')
    else:
      print('sale chor')

In [20]:
obj = Atm()

2884150648784


In [21]:
obj.get_balance()

0

In [22]:
obj.set_balance(1000)

In [24]:
obj.withdraw()

enter the pin 
enter the amount 68568


abe garib


# 9. Collection of Objects

## 9.1 Objects love living in lists and dicts

Real systems create hundreds of objects; the natural next step is storing them in the containers you already know:

```python
p1 = Person('nitish', 'male')
p2 = Person('ankit', 'male')
p3 = Person('ankita', 'female')

L = [p1, p2, p3]               # list of objects
for p in L:
    print(p.name, p.gender)    # iterate and use each object

d = {'p1': p1, 'p2': p2}       # dict of objects
for k in d:
    print(d[k].gender)         # access by key
```

- A **list of objects** suits sequential processing — loops, filters, simulations.
- A **dict of objects** supports fast lookup by id or name.
- Inside the loop every element is a normal object: `p.name`, `p.greet()`.
- List comprehensions work too: `[p.name for p in L]`.
- Sorting by attribute: `sorted(L, key=lambda p: p.name)`.

Storing objects in containers is what turns OOP from "one class" into a working little system.

### Collection of objects

### Key ideas
Classes/objects model state+behavior. Encapsulation hides internals; abstraction focuses on essentials. Inheritance/polymorphism extend reuse.

## Collection of objects

```python
L = [p1, p2, p3]          # list of Person objects
for p in L:
    print(p.name)

d = {'1': p1, '2': p2}    # dict of objects bhi
```

- Keep objects in lists/dicts → use their methods/attributes
- Common: store database/batch data as objects and iterate over them
- During iteration `p.name`, `p.gender` are accessible on every element


## 9.2 Deep dive — structuring many objects

```python
p1, p2, p3 = Person('a','M'), Person('b','F'), Person('c','M')
L = [p1, p2, p3]                # list of objects
for p in L:
    print(p.name, p.gender)     # iterate objects

d = {p.name: p for p in L}      # dict of objects, keyed by name
d['a'].name                     # lookup by key
```

- Real systems hold hundreds of objects → organize them in a structure.
- List of objects: sequential iterate/process — records, simulations.
- Dict of objects: lookup by key (id, name, email).
- The loop accesses each element's attributes and methods directly.
- Comprehension trick: `{p.name: p for p in L}` builds a dict keyed by an attribute.
- `sorted(L, key=lambda p: p.name)` orders objects by an attribute.
- Grouping data and behavior into containers is collection handling — a core daily skill.

The cells below build a `Person` list and loop to print names and genders, then a `Person` dict and loop over its keys — both styles in action.

In [25]:
# list of objects
class Person:

  def __init__(self,name,gender):
    self.name = name
    self.gender = gender

p1 = Person('nitish','male')
p2 = Person('ankit','male')
p3 = Person('ankita','female')

L = [p1,p2,p3]

for i in L:
  print(i.name,i.gender)

nitish male
ankit male
ankita female


In [26]:
# dict of objects
# list of objects
class Person:

  def __init__(self,name,gender):
    self.name = name
    self.gender = gender

p1 = Person('nitish','male')
p2 = Person('ankit','male')
p3 = Person('ankita','female')

d = {'p1':p1,'p2':p2,'p3':p3}

for i in d:
  print(d[i].gender)

male
male
female


# 10. Static Variables vs Instance Variables

## 10.1 Class-level vs object-level state

Every object carries its own copy of **instance variables** — but sometimes a value should be shared by the whole class:

```python
class Atm:
    __counter = 1            # static/class variable — ONE copy for all instances

    def __init__(self):
        self.pin = ''        # instance variable — every object's own copy
        self.cid = Atm.__counter
        Atm.__counter += 1
```

| Kind | Where | Copies | Reach |
|---|---|---|---|
| Instance var | set with `self.` in `__init__` | one per object | `obj.var` |
| Static var | set at class level | one shared | `ClassName.var` |

- Static variables exist once per class; every instance sees the same value.
- Access them with the class name: `Atm.__counter`, even with no object.
- They are perfect for counters (how many objects have been made), config values, and shared constants.
- The pattern above gives every new object the next id: read the counter, store it as `cid`, then bump it.

### Static Variables(Vs Instance variables)

### What/Why
Variables reference objects in memory. Data types determine operations allowed and behavior.

### Semantics
Python is dynamically typed. Assignment binds names to objects; mutability matters.

### Common mistakes
Confusing assignment vs comparison, mutability side effects.


## Static variables (vs instance)

```python
class Atm:
    __counter = 1            # static: class level, sab objects share

    def __init__(self):
        self.pin = ''        # instance: har object ki copy
        self.cid = Atm.__counter
        Atm.__counter += 1
```

- **Instance variable** — every object has its own copy (`self.pin`)
- **Static variable** — class level (`Atm.__counter`), shared by all objects
- Static access: `ClassName.variable` (works even without an object)
- Use-case: counter, config, shared global value


## 10.2 Deep dive — the counting Atm

```python
class Atm:
    __counter = 1           # CLASS variable — shared by ALL instances

    def __init__(self):
        self.cid = Atm.__counter   # instance var: my serial number
        Atm.__counter += 1
```

- **Static/class variable**: class-level, one copy shared by everyone.
- **Instance variable**: each object's own separate copy.
- Use statics for counters, config, shared state.
- Access via `ClassName.var` or `self.var` (both reach the same slot) — statics work even with NO instance.
- Auto-increment pattern: read the shared counter, assign it to the new object as `cid`, then bump the counter.

The far-reaching `Atm` class in the cells uses exactly this: it stores a serial id (`cid`) on every object and also exposes a `@staticmethod` `get_counter()`. The saved outputs show `Atm.get_counter()` → 2 and `c3.cid` → 3 — the precise numbers depend on how many objects were created before that snapshot, but the mechanism stays the same: one shared, ever-increasing class counter.

In [27]:
# need for static vars

In [28]:
class Atm:

  __counter = 1

  # constructor(special function)->superpower -> 
  def __init__(self):
    print(id(self))
    self.pin = ''
    self.__balance = 0
    self.cid = Atm.__counter
    Atm.__counter = Atm.__counter + 1
    #self.menu()

  # utility functions
  @staticmethod
  def get_counter():
    return Atm.__counter


  def get_balance(self):
    return self.__balance

  def set_balance(self,new_value):
    if type(new_value) == int:
      self.__balance = new_value
    else:
      print('beta bahot maarenge')

  def __menu(self):
    user_input = input("""
    Hi how can I help you?
    1. Press 1 to create pin
    2. Press 2 to change pin
    3. Press 3 to check balance
    4. Press 4 to withdraw
    5. Anything else to exit
    """)

    if user_input == '1':
      self.create_pin()
    elif user_input == '2':
      self.change_pin()
    elif user_input == '3':
      self.check_balance()
    elif user_input == '4':
      self.withdraw()
    else:
      exit()

  def create_pin(self):
    user_pin = input('enter your pin')
    self.pin = user_pin

    user_balance = int(input('enter balance'))
    self.__balance = user_balance

    print('pin created successfully')

  def change_pin(self):
    old_pin = input('enter old pin')

    if old_pin == self.pin:
      # let him change the pin
      new_pin = input('enter new pin')
      self.pin = new_pin
      print('pin change successful')
    else:
      print('nai karne de sakta re baba')

  def check_balance(self):
    user_pin = input('enter your pin')
    if user_pin == self.pin:
      print('your balance is ',self.__balance)
    else:
      print('chal nikal yahan se')

  def withdraw(self):
    user_pin = input('enter the pin')
    if user_pin == self.pin:
      # allow to withdraw
      amount = int(input('enter the amount'))
      if amount <= self.__balance:
        self.__balance = self.__balance - amount
        print('withdrawl successful.balance is',self.__balance)
      else:
        print('abe garib')
    else:
      print('sale chor')

In [29]:
c1 = Atm()

2884150650128


In [30]:
Atm.get_counter()

2

In [31]:
c3 = Atm()

2884148734672


In [32]:
c3.cid

2

In [33]:
Atm.counter

AttributeError: type object 'Atm' has no attribute 'counter'

# 11. Static Methods

## 11.1 Behavior that needs no object

A **static method** belongs to the class but expects no `self`:

```python
class Lion:
    __water_source = "well in the circus"   # static variable

    @staticmethod
    def get_water_source():                 # static method — no self
        return Lion.__water_source
```

- The `@staticmethod` decorator marks a method that does not receive `self`.
- Call it through the class: `Lion.get_water_source()` — no instance needed.
- Perfect for helper or utility logic that does not depend on any object's state.
- It can still read static variables via `ClassName.var`.

| Method kind | First arg | Needs an instance? |
|---|---|---|
| Instance method | `self` | yes |
| `@staticmethod` | none | no |
| `@classmethod` | `cls` | no (receives the class) |

Static methods are "pure functions grouped with the class" — handy utilities living inside the class's namespace. The cells below show the triple life of every member: instance methods (need `self`), class methods (need `cls` for shared state), and static methods (need nothing) — keep their varargs as the tell that decides which world a method lives in.

### Static methods

##### Points to remember about static

- Static attributes are created at class level.
- Static attributes are accessed using ClassName.
- Static attributes are object independent. We can access them without creating instance (object) of the class in which they are defined.
- The value stored in static attribute is shared between all instances(objects) of the class in which the static attribute is defined.

## Static methods & static reminders

```python
class Lion:
    __water_source = 'well in the circus'   # static private

    def __init__(self, name, gender):
        self.__name = name

    def drinks_water(self):
        print(self.__name, 'drinks from', Lion.__water_source)
```

- Static attributes are created at class level — no `obj` involved
- Access statics with `ClassName.attr`
- Use the private static `__water_source` only from inside the class
- Meaning of the "points to remember" block: object-independent, shared, class scope


## Deep Dive : Static methods

```python
class Lion:
    __water_source = "well in the circus"     # static var
    @staticmethod
    def drink():                             # static method
        print("Drinking from", Lion.__water_source)
```

- `@staticmethod` — a method without `self`
- Called via the class: `Lion.drink()` — no instance needed
- When to use: helper logic that does not depend on state
- A utility living inside the class namespace
- No access to self / instance data
- Can access static vars via the ClassName
- vs instance method — self is required
- vs classmethod `@classmethod` (cls as the first parameter)
- Statics: pure functions grouped with the class
- Point to remember: static attrs are created at class load time


## Summary

This session turned simple classes into a communicating, self-protecting system. You accessed object data with attributes (`p.name`) and behavior with methods (`p.greet()`), learned that reading a missing attribute raises `AttributeError`, and discovered that Python happily lets you *add* attributes from outside for any single object. Reference variables taught you that a name is only a pointer: `q = p` shares one object, functions receive objects by reference, and mutating an object inside a function is visible everywhere the object is known. Encapsulation showed how `__private` attributes guarded by getter/setter methods hide state and add validation. You stored objects inside lists and dicts to iterate and look them up, and finally met class-level members: static variables that count every instance and `@staticmethod` utilities callable without an object. Together these tools let you model systems — points, ATMs, lions, collections — where data stays safe and shared knowledge lives in the right place.

In [34]:
class Lion:
  __water_source="well in the circus"

  def __init__(self,name, gender):
      self.__name=name
      self.__gender=gender

  def drinks_water(self):
      print(self.__name,
      "drinks water from the",Lion.__water_source)

  @staticmethod
  def get_water_source():
      return Lion.__water_source

simba=Lion("Simba","Male")
simba.drinks_water()
print( "Water source of lions:",Lion.get_water_source())

Simba drinks water from the well in the circus
Water source of lions: well in the circus
